In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
from torchvision.datasets import MNIST
from sklearn.decomposition import PCA


transform = transforms.Compose(
    [
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]
)

train_set = MNIST(root='./data', train=True, transform=transform, download=True)
test_set = MNIST(root='./data', train=False, transform=transform, download=False)

device = "cuda"

basic nonlinear autoencoder

In [ ]:
class VanillaAutoEncoder(nn.Module):

    def __init__(self, bottleneck_size=2):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size)
        )

        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 32*32),
            nn.Tanh()
        )

    def forward_enc(self, x):
        return self.encoder(x)

    def forward_dec(self, x):
        x = self.decoder(x)
        x = x.reshape(-1, 1, 32, 32)
        return x

    def forward(self, x):
        batch, channels, height, width = x.shape

        x = x.flatten(1)

        enc = self.forward_enc(x)

        dec = self.forward_dec(enc)

        return enc, dec

super simple training script

In [ ]:
def train(model,
          train_set,
          test_set,
          batch_size,
          training_iterations,
          evaluation_iterations):

    model = model.to(device)

    trainloader = DataLoader(train_set, shuffle=True, batch_size=batch_size)
    testloader = DataLoader(test_set, shuffle=False, batch_size=batch_size)
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    train_loss = []
    evaluation_loss = []
    train_losses = []
    evaluation_losses = []

    encoded_data_per_eval = []

    pbar = tqdm(range(training_iterations))


    train = True
    step_counter = 0
    while train:

        for images, labels in trainloader:
            images = images.to(device)

            encoded, reconstruction = model(images)

            loss = torch.mean((images - reconstruction)**2)
            train_loss.append(loss.item())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()



            if step_counter % evaluation_iterations == 0:
                model.eval()
                encoded_evaluations = []

                for images, labels in testloader:
                    with torch.no_grad():
                        images = images.to(device)
                        encoded, reconstruction = model(images)
                        loss = torch.mean((images - reconstruction)**2)
                        evaluation_loss.append(loss.item())

                        encoded, labels = encoded.cpu().flatten(1), labels.reshape(-1, 1)

                        encoded_evaluations.append(torch.cat((encoded, labels), axis=-1))


                encoded_data_per_eval.append(torch.concatenate(encoded_evaluations).detach())

                train_loss = np.mean(train_loss)
                evaluation_loss = np.mean(evaluation_loss)

                train_losses.append(train_loss)
                evaluation_losses.append(evaluation_loss)

                print("training loss:", train_loss)
                print("eval loss:", evaluation_loss)

                train_loss = []
                evaluation_loss = []

                model.train()

            step_counter += 1
            pbar.update(1)


            if step_counter >= training_iterations:
                print("completed training")
                train = False
                break


    encoded_data_per_eval = [np.array(i) for i in encoded_data_per_eval]

    return model, train_losses, evaluation_losses, encoded_data_per_eval

vanilla_model = VanillaAutoEncoder()
model, train_losses, evaluation_losses, encoded_data_per_eval = train(vanilla_model, train_set, test_set, batch_size=64, training_iterations=2500, evaluation_iterations=250)

plotting the embedding space through training

In [ ]:
def build_embedding_plot(encoding, title):

    encoding = pd.DataFrame(encoding, columns=["x", "y", "class"])
    encoding = encoding.sort_values(by="class")
    encoding["class"] = encoding["class"].astype(int).astype(str)

    for grouper, group in encoding.groupby("class"):

        plt.scatter(x = group["x"], y = group["y"], label = grouper, alpha = 0.8, s = 5)

    plt.title(title)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend()
    plt.show()


In [ ]:
for i in range(len(encoded_data_per_eval)):
    build_embedding_plot(encoded_data_per_eval[i], "ae embedding")

just for fun, a linear autoencoder

although this isn't exactly pca (because we have no orthogonality restrictions on the components), this should look somewhat similar to what pca produces!

In [ ]:
class LinearAutoEncoder(nn.Module):

    def __init__(self, bottleneck_size=2):
        super().__init__()
        self.encoder = nn.Linear(32*32, bottleneck_size)
        self.decoder = nn.Linear(bottleneck_size, 32*32)

    def forward(self, x):

        x = x.flatten(1)
        enc = self.encoder(x)
        dec = self.decoder(enc)
        dec = dec.reshape(-1, 1, 32, 32)

        return enc, dec

In [ ]:
model = LinearAutoEncoder(2)
model, train_losses, evaluation_losses, encoded_data_per_eval = train(model, train_set, test_set, batch_size=64, training_iterations=25000, evaluation_iterations=250)

In [ ]:
build_embedding_plot(encoded_data_per_eval[-1], "linear autoencoder")

let compare to pca!

In [ ]:
data = np.array(torch.cat([d[0].flatten(1) for d in test_set], dim=0))
labels = np.array([d[1] for d in test_set]).reshape(-1, 1)

# pca embeddings
pca = PCA(n_components=2)
pca_encoding = pca.fit_transform(data)
pca_encoding = np.concatenate((pca_encoding, labels), axis=-1)
pca_encoding = pd.DataFrame(pca_encoding, columns=["x", "y", "class"])
pca_encoding = pca_encoding.sort_values(by="class")
pca_encoding["class"] = pca_encoding["class"].astype(int).astype(str)

# grab last linear encodings
linauto_encoding = encoded_data_per_eval[-1]
linauto_encoding = pd.DataFrame(linauto_encoding, columns=["x", "y", "class"])
linauto_encoding = linauto_encoding.sort_values(by="class")
linauto_encoding["class"] = linauto_encoding["class"].astype(int).astype(str)

# lets plot it all now!
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# plot pca plot
for grouper, group in pca_encoding.groupby("class"):
    ax1.scatter(x = group["x"], y = group["y"], label = grouper, alpha = 0.8, s = 5)
ax1.legend()
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_title("pca embedding")

# plot linear autoencoder plot (BU KISIM EKSİKTİ)
for grouper, group in linauto_encoding.groupby("class"):
    ax2.scatter(x = group["x"], y = group["y"], label = grouper, alpha = 0.8, s = 5)
ax2.legend()
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_title("linear autoencoder")

plt.show()

In [ ]:
pc_1, pc_2 = torch.tensor(pca.components_[0]), torch.tensor(pca.components_[1])

torch.rad2deg(torch.acos(torch.dot(pc_1, pc_2)))

torch.norm(pc_1)

In [ ]:
model_weights = model.encoder.weight

wv_1, wv_2 = model_weights[0].cpu(), model_weights[1].cpu()

wv_1 = wv_1 / torch.norm(wv_1)
wv_2 = wv_2 / torch.norm(wv_2)

torch.rad2deg(torch.acos(torch.dot(wv_1, wv_2)))

convolutional autoencoder

In [ ]:
rand = torch.randn(2, 3, 64, 64)

conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, stride=2, padding=1)
tconv = nn.ConvTranspose2d(in_channels=3, out_channels=3, kernel_size=3, stride=2, padding=1)

tconv(conv(rand)).shape

In [ ]:
rand_tensor_1 = torch.rand(2, 3, 64, 64)
rand_tensor_2 = torch.rand(2, 3, 63, 63)

print(conv(rand_tensor_1).shape)
print(conv(rand_tensor_2).shape)

In [ ]:
class ConvolutionalAutoEncoder(nn.Module):

    def __init__(self, in_channels = 1, channels_bottleneck = 4):
        super().__init__()
        self.bottleneck = channels_bottleneck
        self.in_channels = in_channels

        self.encoder_conv = nn.Sequential(

            nn.Conv2d(in_channels=self.in_channels, out_channels=8, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(in_channels=16, out_channels=self.bottleneck, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.bottleneck),
            nn.ReLU(),

        )

        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(in_channels = self.bottleneck, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels = 16, out_channels=8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.ConvTranspose2d(in_channels = 8, out_channels=self.in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Tanh(),

        )


    def forwrad_enc(self, x):
        return self.encoder_conv(x)


    def forward_dec(self, x):
        return self.decoder_conv(x)


    def forward(self, x):
        batch_size, num_channels, height, width = x.shape
        enc = self.forwrad_enc(x)
        dec = self.forward_dec(enc)

        return enc, dec



In [ ]:
conv_model = ConvolutionalAutoEncoder()
conv_model, train_losses, evaluation_losses, conv_encoded_data_per_eval = train(conv_model,
                                                                                train_set,
                                                                                test_set,
                                                                                batch_size=64,
                                                                                training_iterations=25000,
                                                                                evaluation_iterations=250)

In [ ]:
from sklearn.manifold import TSNE
conv_ae_encoding = conv_encoded_data_per_eval[-1]


conv_ae_features =  conv_ae_encoding[:, :-1]
labels = conv_ae_encoding[:, -1]
tsne = TSNE(2, n_jobs=-1)
conv_ae_compressed = tsne.fit_transform(X = conv_ae_features)


conv_ae_encoding = np.hstack((conv_ae_compressed, labels.reshape(-1, 1)))
conv_ae_encoding = pd.DataFrame(conv_ae_encoding, columns=["x", "y", "class"])
conv_ae_encoding = conv_ae_encoding.sort_values(by="class")
conv_ae_encoding["class"] = conv_ae_encoding["class"].astype(int).astype(str)


for grouper, group in conv_ae_encoding.groupby("class"):
    plt.scatter(x = group["x"], y = group["y"], label = grouper, alpha = 0.8, s = 5)

what about generating new images

In [ ]:
generated_index = 0
image, label = test_set[generated_index]

# Ensure both models are on the correct device
vanilla_model = vanilla_model.to(device)
conv_model = conv_model.to(device)

# Move input to device and perform inference
input_tensor = image.unsqueeze(0).to(device)

_, vanilla_reconstructed = vanilla_model(input_tensor)
_, conv_reconstructed = conv_model(input_tensor)

# Move results back to CPU for plotting
vanilla_reconstructed = vanilla_reconstructed.to("cpu").detach().numpy()
conv_reconstructed = conv_reconstructed.to("cpu").detach().numpy()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].imshow(image.squeeze(), cmap="gray")
ax[0].set_title("original image")
ax[0].axis("off")

ax[1].imshow(vanilla_reconstructed.squeeze(), cmap="gray")
ax[1].set_title("vanilla autoencoder")
ax[1].axis("off")

ax[2].imshow(conv_reconstructed.squeeze(), cmap="gray")
ax[2].set_title("conv autoencoder")
ax[2].axis("off")

plt.show()

what about generating new images!

In [ ]:
final_embeddings = encoded_data_per_eval[-1]

build_embedding_plot(final_embeddings, "ae embedding")
avg_digit_embeddings = []
for i in range(10):

    avg_embeddings = np.mean(final_embeddings[final_embeddings[:, 2] == i][:, :2], axis=0)
    avg_digit_embeddings.append(avg_embeddings)

avg_digit_embeddings = torch.tensor(np.array(avg_digit_embeddings))
pred_images = vanilla_model.forward_dec(avg_digit_embeddings.to("cuda"))


fix, axes = plt.subplots(1, 10, figsize = (15, 5))
for idx, img in enumerate(pred_images):
    img = img.squeeze().detach().cpu().numpy()
    axes[idx].imshow(img, cmap="gray")
    axes[idx].set_xticklabels([])
    axes[idx].set_yticklabels([])
    axes[idx].axis("off")

fig.subplots_adjust(wspace=0, hspace=0)

plt.show()

In [ ]:
def interpolate_space(model, x_range=(-25, 10), y_range=(-20, 5), num_steps=20):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    x_space = np.linspace(x_range[0], x_range[1], num_steps)
    y_space = np.linspace(y_range[0], y_range[1], num_steps)

    points = []
    for x in x_space:
        for y in y_space:
            points.append([x, y])

    points = torch.tensor(points, dtype=torch.float32).to(device)

    # pass through model decoder and reshape
    dec = model.forward_dec(points).detach().cpu()
    dec = dec.reshape((num_steps, num_steps, *dec.shape[1:]))

    fig, ax = plt.subplots(num_steps, num_steps, figsize=(12, 12))

    for x in range(num_steps):
        for y in range(num_steps):
            img = np.array(dec[x, y].permute(1, 2, 0))
            ax[x, y].imshow(img, cmap="gray")
            ax[x, y].set_xticklabels([])
            ax[x, y].set_yticklabels([])
            ax[x, y].axis("off")

    fig.subplots_adjust(wspace=0, hspace=0)

    plt.show()

interpolate_space(vanilla_model)